In [2]:
!pip install gliner pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46

In [ ]:
import os

# Create the main data directory
os.makedirs("data", exist_ok=True)

# Create the chunking_tests subdirectory
os.makedirs("data/chunking_tests", exist_ok=True)

print("Directories successfully created:")
print(f"- {os.path.abspath('data')}")
print(f"- {os.path.abspath('data/chunking_tests')}")

print("***** Now user load emails.csv into data file *****")

In [ ]:
import pandas as pd
import email
import re
import spacy

# Load spaCy's small English model
nlp = spacy.load("en_core_web_sm")

def get_text_from_email(msg):
    """Extract the plain text content from an email message."""
    parts = []
    for part in msg.walk():
        if part.get_content_type() == 'text/plain':
            parts.append(part.get_payload())
    return ''.join(parts)

def split_email_addresses(line):
    """
    Separate multiple email addresses and strip extra whitespace.
    If there's only one address, return it as a string.
    If there are multiple, return them as a comma-separated string.
    """
    if line:
        addrs = list(map(lambda x: x.strip(), line.split(',')))
        return addrs[0] if len(addrs) == 1 else ', '.join(addrs)
    else:
        return None

def clean_text(text):
    """
    Clean and preprocess text:
      - Convert to lowercase.
      - Remove forwarded message markers, email headers in the body,
        message IDs, timestamps, URLs, and email addresses.
      - Remove unwanted special characters and normalize whitespace.
    """
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r'---+ ?forwarded by.+?---+', ' ', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'---+ ?original message.+?---+', ' ', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'---+ ?forwarded message.+?---+', ' ', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'from:.*?(?=\n\n|\n\w)', ' ', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'sent:.*?(?=\n\n|\n\w)', ' ', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'to:.*?(?=\n\n|\n\w)', ' ', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'subject:.*?(?=\n\n|\n\w)', ' ', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'cc:.*?(?=\n\n|\n\w)', ' ', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'bcc:.*?(?=\n\n|\n\w)', ' ', text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r'message-id:.*', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'date:.*', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'content-transfer-encoding:.*', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'http[s]?://(?:[a-zA-Z0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', ' ', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', ' ', text)
    text = re.sub(r'[^a-zA-Z\s.,!?-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize_text(text):
    """
    Tokenize text using spaCy and apply lemmatization.
    Only tokens that are alphabetic or punctuation are included.
    """
    if not text or pd.isna(text):
        return []
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if token.is_alpha or token.is_punct]
    return tokens

def preprocess_data(input_file, output_file, max_rows=None):
    """
    Preprocess the email data:
      - Read the CSV with a "message" column containing raw email text.
      - Parse each email to extract headers and plain text content.
      - Clean and tokenize the subject and content.
      - Process the email addresses in the From and To fields.
      - Drop any rows with empty cleaned content.
      - Return a DataFrame with the columns:
            To, From, X-To, X-From, content_clean, subject_clean, content_tokens, subject_tokens.
    """
    # Load the dataset and limit rows if necessary
    emails_df = pd.read_csv(input_file)
    if max_rows:
        emails_df = emails_df.head(max_rows)
    print(f"Processing {len(emails_df)} emails")

    # Parse the raw email messages from the 'message' column
    messages = list(map(email.message_from_string, emails_df['message']))

    # Build records from parsed messages
    records = []
    for msg in messages:
        record = {}
        # Extract header fields; default to empty string if missing
        record['subject'] = msg['Subject'] if msg['Subject'] is not None else ''
        record['From'] = msg['From'] if msg['From'] is not None else ''
        record['To'] = msg['To'] if msg['To'] is not None else ''
        record['X-To'] = msg['X-To'] if msg['X-To'] is not None else ''
        record['X-From'] = msg['X-From'] if msg['X-From'] is not None else ''
        # Extract plain text content
        record['content'] = get_text_from_email(msg)
        records.append(record)

    # Create DataFrame from the records
    result_df = pd.DataFrame(records)

    # Process email addresses for From and To fields
    result_df['From'] = result_df['From'].apply(split_email_addresses)
    result_df['To'] = result_df['To'].apply(split_email_addresses)

    # Clean and tokenize the subject and content
    result_df['subject_clean'] = result_df['subject'].apply(clean_text)
    result_df['subject_tokens'] = result_df['subject_clean'].apply(tokenize_text)
    result_df['content_clean'] = result_df['content'].apply(clean_text)
    result_df['content_tokens'] = result_df['content_clean'].apply(tokenize_text)

    # Select the desired columns
    final_df = result_df[['To', 'From', 'X-To', 'X-From',
                          'content_clean', 'subject_clean',
                          'content_tokens', 'subject_tokens']]

    # Drop rows with empty cleaned content
    final_df = final_df[final_df['content_clean'] != ""]

    # Save the processed DataFrame to CSV
    final_df.to_csv(output_file, index=False)
    print(f"Saved processed data to {output_file}")
    return final_df

if __name__ == "__main__":
    processed_df = preprocess_data("data/emails.csv", "data/filtered.csv", max_rows=1000)

In [6]:
import pandas as pd
from gliner import GLiNER
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
df = pd.read_csv('data/filtered.csv')
# df = df[0:20]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")
model.to(device)

labels = ["date", "location", "person", "action", "finance", "legal", "event", "product", "organization"]

# potential chunk sizes and overlap sizes for automated testing
chunk_sizes = [300, 400, 500, 600]
overlap_sizes = [100, 150, 200, 250]

results = []

def print_gpu_utilization():
    if torch.cuda.is_available():
        print(f"GPU utilization: {torch.cuda.memory_allocated()/1024**2:.2f} MB / {torch.cuda.get_device_properties(0).total_memory/1024**2:.2f} MB")
    else:
        print("No GPU available")

"""
Extracting Named Entities with GLiNER
"""
def apply_gliner_labeling(row, chunk_size):
    text = str(row['content_clean'])

    chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
    all_entities = []

    for chunk in chunks:
        try:
            # print_gpu_utilization()
            entities = model.predict_entities(chunk, labels, threshold=0.5)
            all_entities.extend(entities)
        except Exception as e:
            print(f"Error processing chunk: {e}")

    entity_dict = {}
    for entity in all_entities:
        entity_type = entity['label']
        entity_text = entity['text']
        if entity_type not in entity_dict:
            entity_dict[entity_type] = []
        entity_dict[entity_type].append(entity_text)
    return entity_dict


"""
Smart Chunking that Avoids Entity Splitting
"""
def ner_chunking(text, entities, chunk_size, overlap_size):
    chunks = []
    start_idx = 0

    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))

        # expand chunk to prevent cutting words in half
        # ensure we're not at the last chunk
        if end_idx < len(text):
            # extend until reaching a non-alphanumeric char
            while end_idx < len(text) and text[end_idx].isalnum():
                end_idx += 1

        # making sure named entities aren't split across chunks
        for entity_list in entities.values():
            # print_gpu_utilization()
            for entity in entity_list:
                entity_start = text.find(entity)
                entity_end = entity_start + len(entity)

                # if entity starts near the chunk end, extend the chunk to include it
                if entity_start < end_idx and entity_end > end_idx:
                    end_idx = entity_end

        # adding chunk and move index
        chunks.append(text[start_idx:end_idx])
        start_idx += (chunk_size - overlap_size)

    return chunks

"""
Calculate TF-IDF Scores for Each Chunk
"""
def tf_idf_calc(chunks, top_n):
    tf_idf_results = []

    for chunk in chunks:
        if not chunk:
            tf_idf_results.append([])
            continue

        vectorizer = TfidfVectorizer(stop_words='english')

        try:
            tf_idf_matrix = vectorizer.fit_transform([chunk])
            feature_names = vectorizer.get_feature_names_out()

            # getting scores
            scores = zip(feature_names, tf_idf_matrix.toarray().flatten())
            sorted_scores = sorted(scores, key=lambda x: x[1], reverse=True)

            top_words = [word for word, score in sorted_scores[:top_n]]
            tf_idf_results.append(top_words)
        except ValueError:
            # in case of an empty vocabulary
            tf_idf_results.append([])

    return tf_idf_results

"""
Evaluate Chunking Quality Using Entity Consistency Score
"""
def compute_entity_consistency(df):
    entity_scores = []

    for _, row in df.iterrows():
        entities = row['named_entities']
        chunks = row['ner_chunks']
        retained_count = 0
        counted_entities = set()

        # totalling entity occurrences
        entity_count = sum(len(v) for v in entities.values())
        # print(f"entity_count: {entity_count}")
        for v in entities.values():
            for ent in v:
                # print(f"ent: {ent}\n")
                if ent not in counted_entities:
                    for chunk in chunks:
                        # print(f"chunk: {chunk}\n")
                        if str(ent) in str(chunk):
                            # print("***** found ent: {ent} ***** \n")
                            retained_count += 1
                            counted_entities.add(ent)
                            break
        # print(f"retained_count: {retained_count}")

        # ECS: % of entities retained in chunks
        ecs = round((retained_count / entity_count), 2) if entity_count > 0 else 0
        entity_scores.append(ecs)

    df['entity_consistency_score'] = entity_scores
    return df

"""
Evaluate Overlap Between GLiNER Entities and Top TF-IDF Terms
"""
def compute_entity_tf_idf_overlap(df):
    overlap_scores = []

    for _, row in df.iterrows():
        entities = set([ent for entity_list in row['named_entities'].values() for ent in entity_list])
        tf_idf_terms = set([term for chunk in row['tf_idf'] for term in chunk])

        common_terms = entities.intersection(tf_idf_terms)
        overlap_score = round((len(common_terms) / len(entities)),2) if len(entities) > 0 else 0

        overlap_scores.append(overlap_score)

    df['entity_tf_idf_overlap_score'] = overlap_scores
    return df

"""
Applying NER-Based Chunking & TF-IDF Scoring to the Dataset
"""
def process_dataframe(df, chunk_size, overlap_size, top_n):
    print(f"Running chunk_size={chunk_size}, overlap={overlap_size}...")
    df['named_entities'] = df.apply(lambda row: apply_gliner_labeling(row, chunk_size), axis=1)
    df['ner_chunks'] = df.apply(lambda row: ner_chunking(str(row['content_clean']), row['named_entities'], chunk_size, overlap_size), axis=1)
    df['tf_idf'] = df.apply(lambda row: tf_idf_calc(row['ner_chunks'], top_n), axis=1)
    df = compute_entity_consistency(df)
    df = compute_entity_tf_idf_overlap(df)
    avg_ecs = df['entity_consistency_score'].mean()
    avg_overlap = df['entity_tf_idf_overlap_score'].mean()
    results.append((chunk_size, overlap_size, avg_ecs, avg_overlap))
    df.to_csv(f"data/chunking_tests/emails_chunk_{chunk_size}_overlap_{overlap_size}.csv", index=False)

# testing chunk sizes and overlap sizes for the best combo
"""
for chunk_size in chunk_sizes:
    for overlap_size in overlap_sizes:
        process_dataframe(df.copy(), chunk_size, overlap_size, top_n=5)

# finding the best chunk size based on avg ECS and overlap scores
results_df = pd.DataFrame(results, columns=["Chunk Size", "Overlap Size", "Avg ECS", "Avg TF-IDF Overlap"])
results_df.to_csv("data/chunking_evaluation_results.csv", index=False)

best_config = results_df.sort_values(by=["Avg ECS", "Avg TF-IDF Overlap"], ascending=False).iloc[0]
best_chunk_size = int(best_config["Chunk Size"])
best_overlap_size = int(best_config["Overlap Size"])
print(f"\nBest Chunk Size: {best_chunk_size}, Best Overlap Size: {best_overlap_size}")

# saving the best chunking res
best_df = pd.read_csv(f"data/chunking_tests/emails_chunk_{best_chunk_size}_overlap_{best_overlap_size}.csv")
"""
process_dataframe(df.copy(), 300, 250, top_n=5)
best_df = pd.read_csv(f"data/chunking_tests/emails_chunk_300_overlap_250.csv")
best_df.to_csv("data/best_chunking_strategy.csv", index=False)
print("\nBest chunking strategy saved in 'data/best_chunking_strategy.csv'")

Using device: cuda


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/convert_slow_tokenizer.py:561: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Running chunk_size=300, overlap=250...

Best chunking strategy saved in 'data/best_chunking_strategy.csv'


In [14]:
# Try installing packages using Python's pip module directly
import pip

def install(package):
    try:
        pip.main(['install', package])
        print(f"Successfully installed {package}")
    except Exception as e:
        print(f"Error installing {package}: {e}")

# Install required packages one by one
packages = [
    'langchain',
    'langchain_huggingface',
    'langchain_community',
    'langchain-groq',
    'faiss-cpu',
    'transformers',
    'accelerate',
    'sentence-transformers',
    'faiss-gpu'
]

for package in packages:
    install(package)

Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.


Requirement already satisfied: langchain in /usr/local/lib/python3.11/dist-packages (0.3.21)

Requirement already satisfied: langchain-core<1.0.0,>=0.3.45 in /usr/local/lib/python3.11/dist-packages (from langchain) (0.3.47)

Requirement already satisfied: langchain-text-splitters<1.0.0,>=0.3.7 in /usr/local/lib/python3.11/dist-packages (from langchain) (0.3.7)

Requirement already satisfied: langsmith<0.4,>=0.1.17 in /usr/local/lib/python3.11/dist-packages (from langchain) (0.3.15)

...

Requirement already satisfied: urllib3<3,>=1.21.1 in /usr/local/lib/python3.11/dist-packages (from requests->huggingface-hub>=0.20.0->sentence-transformers) (2.3.0)

Requirement already satisfied: certifi>=2017.4.17 in /usr/local/lib/python3.11/dist-packages (from requests->huggingface-hub>=0.20.0->sentence-transformers) (2025.1.31)

Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.


ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)


ERROR: No matching distribution found for faiss-gpu


In [18]:
import pandas as pd
import numpy as np
import torch
import faiss
import os
import ast
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains.retrieval import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"]= "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"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.makedirs("faiss_index", exist_ok=True)

df = pd.read_csv('data/best_chunking_strategy.csv')
new_dataframe = True
test_questions = [
    "What does randy need to send a schedule of?",
    "What are some of randy's action items?",
    "What is Philip's proposal focused on, and can you provided details about the proposal?",
    "Can you provide me more detail about the microturbine power generation deal?"
]

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")


if torch.cuda.is_available():
    try:
        import faiss
        if not hasattr(faiss, 'StandardGpuResources'):
            raise ImportError("FAISS without GPU support")
    except ImportError:
        import pip
        pip.main(['install', 'faiss-gpu'])
        print("Installed FAISS GPU version")


"""
Creating Vector Database
"""
def create_vector_database(df, save_path="faiss_index"):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model_kwargs = {'device': device}
    print(f"Using device: {device} for embeddings")
    encode_kwars = {'normalize_embeddings': True}
    embeddings_model = HuggingFaceEmbeddings(
        model_name="intfloat/e5-base-v2",
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwars,
    )

    all_chunks = []
    all_metadata = []

    for index, row in df.iterrows():
        raw_chunks = row['ner_chunks']
        chunks_array = ast.literal_eval(raw_chunks)

        for i, chunk in enumerate(chunks_array):
            all_chunks.append(chunk)
            all_metadata.append({
                "source_row": index,
                "chunk_index": i
            })

    vector_database = FAISS.from_texts(
        texts=all_chunks,
        embedding=embeddings_model,
        metadatas=all_metadata
    )

    vector_database.save_local(save_path)
    print(f"Vector database created and saved to {save_path}")
    return vector_database, embeddings_model

"""
Loading Vector Database
"""
def load_vector_database(save_path="faiss_index"):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model_kwargs = {'device': device}
    print(f"Using device: {device} for embeddings")
    encode_kwars = {'normalize_embeddings': True}
    embeddings_model = HuggingFaceEmbeddings(
        model_name="intfloat/e5-base-v2",
        model_kwargs=model_kwargs,
        encode_kwargs=encode_kwars,
    )

    if os.path.exists(save_path) and os.path.isdir(save_path):
        vector_database = FAISS.load_local(save_path, embeddings_model, allow_dangerous_deserialization=True)
        return vector_database, embeddings_model
    else:
        raise FileNotFoundError(f"No vector database found at {save_path}")


"""
Creating LLM Model: Llama3 8B
"""
llama_llm = ChatGroq(model_name="llama3-8b-8192")


"""
Creating Prompt Template for LLM
"""
#
prompt = ChatPromptTemplate.from_template(
        """
            Answer question only provided the context. Give a detailed answer IN minimum 5 sentences!
            SAY I DONT KNOW IF CONTEXT IS NOT ENOUGH. DONT MAKE UP ANSWERS. BUT YOU ARE FREE TO INFER/SUGGEST.
            {context}

            Here is question:
            {input}
        """
)


"""
Running Queries Through Created RAG LLM + Inspecting cosine similarity score
"""
def query_system(question, new_df):
    database_path = "faiss_index"
    index_file = os.path.join(database_path, "index.faiss")
    if not os.path.exists(database_path) or not os.path.isdir(database_path) or new_df:
        print("Creating vector database for the first time")
        os.makedirs(database_path, exist_ok=True)
        vector_database, embedding_model = create_vector_database(df, database_path)
    else:
        print("Loading existing vector database")
        vector_database, embedding_model = load_vector_database(database_path)

    # displaying top 15 number of cosine similarity matches (raw retrieval)
    print(f"\nInspecting cosine similarity scores for: \"{question}\"\n")
    query_embedding = embedding_model.embed_query(question)
    results = vector_database.similarity_search_with_score_by_vector(
        embedding=query_embedding,
        k=15
    )

    for i, (doc, score) in enumerate(results):
        print(f"Top-{i+1} Chunk Cosine Score = {score:.4f}")
        print(f"Content: {doc.page_content[:100]}...\n")

    # using MMR-based retrieval to pick 10 chunks for the LLM
    retriever = vector_database.as_retriever(
        search_kwargs={'k': 10, 'search_type': 'mmr', 'lambda_mult': 0.5}
    )
    document_chain = create_stuff_documents_chain(llama_llm, prompt)
    retrieval_chain = create_retrieval_chain(retriever, document_chain)

    print("Running MMR-based retrieval and querying the LLM...\n")
    result = retrieval_chain.invoke({"input": question})
    print('========================================')
    print(f"Query: {question}")
    print(f"Answer: {result['answer']}")
    print('========================================\n')
    # recalculating cosine similarity for each retrieved chunks (actual input to LLM)
    print("Retrieved Chunks with Cosine Similarity:\n")
    retrieved_docs = result.get('context', [])
    for i, doc in enumerate(retrieved_docs):
        doc_embedding = embedding_model.embed_documents([doc.page_content])[0]
        cosine_score = float(np.dot(query_embedding, doc_embedding))
        print(f"Chunk {i+1} Cosine Score = {cosine_score:.4f}")
        print(f"Content: {doc.page_content[:100]}...\n")

    return result


for question in test_questions:
    result = query_system(question, new_dataframe)

CUDA available: True
GPU device: Tesla T4


Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.


ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)


ERROR: No matching distribution found for faiss-gpu


Installed FAISS GPU version
Creating vector database for the first time
Using device: cuda for embeddings


...

Query: What is Philip's proposal focused on, and can you provided details about the proposal?
Answer: Based on the provided context, it appears that Philip is proposing to work on a project and is seeking advice and feedback from someone, possibly Patti S, before moving forward. The context suggests that Philip has already discussed the project with someone and is now seeking additional input and guidance.

The key phrases that support this interpretation are:

* "as I mentioned I would be interested in speaking to you as an advisor or at least a sounding board for the key issues"
* "please email or call" - indicating that Philip is seeking a direct communication with the person
* "developments? phillip" - implying that Philip wants to discuss the project's progress and potential changes
* "your thoughts on any changes that need to be made. patti s for example" - suggesting that Philip is open to feedback and willing to consider changes based on the input he receives

Without more spec

Load pretrained SentenceTransformer: intfloat/e5-base-v2

Vector database created and saved to faiss_index

Inspecting cosine similarity scores for: "Can you provide me more detail about the microturbine power generation deal?"

Top-1 Chunk Cosine Score = 0.2629
Content: power generation deal for a national accounts customer, i am developing a proposal to sell power to ...

Top-2 Chunk Cosine Score = 0.2792
Content: i need a corresponding term gas price for same. microturbine is an onsite generation product develop...

Top-3 Chunk Cosine Score = 0.2812
Content: ur phone conversation, in a parallon microturbine power generation deal for a national accounts cust...

Top-4 Chunk Cosine Score = 0.3135
Content: omer, i am developing a proposal to sell power to customer at fixed or collar floor price. to do so ...

Top-5 Chunk Cosine Score = 0.3260
Content: or , , , and years for annual seasonal supply to microturbines to generate fixed kwh for customer. w...

Top-6 Chunk Cosine Score = 0.3289
Content: icroturbines to generate fixed kwh for custome

HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"

Query: Can you provide me more detail about the microturbine power generation deal?
Answer: I don't have enough context to answer this question. The provided text appears to be a conversation between two individuals discussing a power generation deal for a national accounts customer, and it is unclear what specific information is being requested about the microturbine power generation deal. 

If you could provide more context or clarify what you would like to know about the microturbine power generation deal, I would be happy to try and assist you.

Retrieved Chunks with Cosine Similarity:

Chunk 1 Cosine Score = 0.8686
Content: power generation deal for a national accounts customer, i am developing a proposal to sell power to ...

Chunk 2 Cosine Score = 0.8604
Content: i need a corresponding term gas price for same. microturbine is an onsite generation product develop...

Chunk 3 Cosine Score = 0.8594
Content: ur phone conversation, in a parallon microturbine power generation deal for